<a href="https://www.kaggle.com/code/amoghpanthangi/spml2c?scriptVersionId=306034713" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **NEWS CATEGORY CLASSIFICATION**

## **REQUIREMENTS**

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt')
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
lm=WordNetLemmatizer()
import sklearn
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

In [ ]:
!pip install transformers datasets evaluate

## **MODELS**

In [5]:
class RnnModel(nn.Module):
    def __init__(self, vocab_size):
        super(RnnModel, self).__init__()

        self.embed = nn.Embedding(vocab_size, 200)
        self.dropout = nn.Dropout(0.4)
        self.RNN = nn.GRU(200, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3) 
        self.norm = nn.LayerNorm(512) 
        self.output = nn.Linear(512, 42)

    def forward(self, x):
        x=self.embed(x)
        _, hidden = self.RNN(x)
        x=self.dropout(x)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = self.norm(hidden)
        output = self.output(hidden.squeeze(0))
        return output

class LstmModel(nn.Module):
    def __init__(self, vocab_size):
        super(LstmModel, self).__init__()

        self.embed = nn.Embedding(vocab_size, 200)
        self.dropout = nn.Dropout(0.4)
        self.LSTM = nn.LSTM(200, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3) #bidirectional LSTM layer
        self.norm = nn.LayerNorm(512)
        self.output = nn.Linear(512, 42)

    def forward(self, x):
        x=self.embed(x)
        x=self.dropout(x)
        _, (hidden, _) = self.LSTM(x)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = self.norm(hidden)
        output = self.output(hidden.squeeze(0))
        return output

## **TEXT PREPROCESSING**

 ### **CLEANING**

In [6]:
df = pd.read_json('/kaggle/input/news-category-dataset/News_Category_Dataset_v3.json', lines=True)
df['combine'] = df['headline'] + ' : ' + df['short_description'] 
df = df[['category', 'combine']] 

sentences=[]
for s in df['combine']:
    s = s.lower()
    tokens = word_tokenize(s)
    prep_token = []
    for word in tokens:                              
        if word not in stop_words:
            prep_token.append(lm.lemmatize(word))
    sentences.append(prep_token)
    
        
vocab = {'<PAD>':0, '<UNK>':1}
ID=2
for sentence in sentences:
    for word in sentence:
        if word not in vocab:                 
            vocab[word] = ID
            ID+=1

### **ENCODING**

In [7]:
def encode(sentence, vocab):
    encoder=[]
    for word in sentence:
        if word not in vocab:
            encoder.append(vocab['<UNK>'])     
        else:
            encoder.append(vocab[word])
    return encoder     

def padding(sentence, lenmax=75):
    if len(sentence)<lenmax:
        sentence = sentence + [vocab['<PAD>']]*(lenmax-len(sentence))     
    else:
        sentence = sentence[:lenmax]
    return sentence
    
sentences_ = []
for sentence in sentences:
    encoded_sentence = encode(sentence, vocab)
    padded_sentence = padding(encoded_sentence, lenmax=75)
    sentences_.append(padded_sentence)

### **DATASET CREATION**

In [8]:
labels = le.fit_transform(df['category'])
input_tensor = torch.tensor(sentences_, dtype=torch.long)    
label_tensor = torch.tensor(labels, dtype=torch.long)

class NewsCategory(Dataset):
    def __init__(self, inputs, labels ):
        self.inputs = inputs  
        self.labels = labels                
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, ID):
        return self.inputs[ID], self.labels[ID]

In [9]:
full_dset = NewsCategory(input_tensor, label_tensor)
train_size = int(0.8*len(full_dset))
val_size = int(0.1*len(full_dset))
test_size = len(full_dset) - train_size - val_size
train_dset, val_dset, test_dset = random_split(full_dset, [train_size, val_size, test_size])

train_dloader = DataLoader(train_dset, batch_size = 64, shuffle=True)
val_dloader = DataLoader(val_dset, batch_size = 64, shuffle=False)
test_dloader = DataLoader(test_dset, batch_size = 64, shuffle=False)

### **TRAINING AND VALIDATION LOOP**

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rnn_model = RnnModel(vocab_size=len(vocab)).to(device)
lstm_model = LstmModel(vocab_size=len(vocab)).to(device)
loss_fn = nn.CrossEntropyLoss()
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
epochs = 10

In [11]:
def train_val(model, train_dloader, val_dloader, loss_fn, optimizer, device):
    model.train()
    total_train_loss = 0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_dloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    avg_train_loss = total_train_loss / len(train_dloader)
    train_acc = (correct_train / total_train)*100

    model.eval()
    total_val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.inference_mode():
        for inputs, labels in val_dloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = total_val_loss / len(val_dloader)
    val_acc = (correct_val / total_val)*100

    return avg_train_loss, train_acc, avg_val_loss, val_acc

In [ ]:
history = {
    "rnn_acc": [], "rnn_val_acc": [],
    "lstm_acc": [], "lstm_val_acc": []
}

for epoch in range(epochs):
    r_train_loss, r_train_acc, r_val_loss, r_val_acc = train_val(rnn_model, train_dloader, val_dloader, loss_fn, rnn_optimizer, device)
    l_train_loss, l_train_acc, l_val_loss, l_val_acc = train_val(lstm_model, train_dloader, val_dloader, loss_fn, lstm_optimizer, device)
    history["rnn_acc"].append(r_train_acc)
    history["rnn_val_acc"].append(r_val_acc)
    history["lstm_acc"].append(l_train_acc)
    history["lstm_val_acc"].append(l_val_acc)
    print(f"Epoch {epoch+1}")
    print(f"RNN : Train Loss={r_train_loss:.2f}, Train Accuracy={r_train_acc:.2f}, Val loss={r_val_loss:.2f}, Val Accuracy={r_val_acc:.2f}")
    print(f"LSTM : Train Loss={l_train_loss:.2f}, Train Accuracy={l_train_acc:.2f}, Val loss={l_val_loss:.2f}, Val Accuracy={l_val_acc:.2f}")



In [ ]:
epochs_range = range(1, epochs + 1)
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(epochs_range, history["rnn_acc"], label='Train', linewidth=2)
plt.plot(epochs_range, history["rnn_val_acc"], label='Val', linewidth=2)
plt.title('RNN: Train vs Val Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(epochs_range, history["lstm_acc"], label='Train', color='green', linewidth=2)
plt.plot(epochs_range, history["lstm_val_acc"], label='Val', color='lightgreen', linewidth=2)
plt.title('LSTM: Train vs Val Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(epochs_range, history["rnn_val_acc"], label='RNN Val Acc', color='red', linewidth=2)
plt.plot(epochs_range, history["lstm_val_acc"], label='LSTM Val Acc', color='green', linewidth=2)
plt.title('Validation Accuracy: RNN vs LSTM', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()